This notebook is one of multiple notebooks in the `nlp/` section of this project that is specifically directed towards a specialized NLP task, and goes through, in detail, how to complete that task using NLP techniques.

This particular notebook will specifically focus on **text classification** via the **BERT** model.

# Text classification
- Text classification involves assigning predefined categories to text documents, such as sentiment analysis (happy, sad, angry, etc), topic classification (article, novel, pamphlets), or spam detection.
- For this, we are going to use Google's **BERT** model.
- BERT is an encoder-only model and is the first model to effectively implement deep bidirectionality to learn richer representations of the text by attending to words on both sides.
  - It uses WordPiece tokenization to generate token vector embeddings of the inputted text.
  - To tell the difference between sentences, `[SEP]` tokens are added to separate adjacent sentences. 
  - A special `[CLS]` token is added to the beginning of every sequence of text. 
  - The final output with the `[CLS]` token is used as the input to the classification head for classification tasks.
  - BERT also adds a segment embedding to denote whether a token belongs to the first or second sentence in a pair of sentences.
  - BERT is pretrained with two objectives: masked language modeling and next-sentence prediction. In masked language modeling, some percentage of the input tokens are randomly masked, and the model needs to predict these. This solves the issue of bidirectionality, where the model could cheat and see all the words and “predict” the next word. The final hidden states of the predicted mask tokens are passed to a feedforward network with a softmax over the vocabulary to predict the masked word.
  - The second pretraining object is next-sentence prediction. The model must predict whether sentence B follows sentence A. Half of the time sentence B is the next sentence, and the other half of the time, sentence B is a random sentence. The prediction, whether it is the next sentence or not, is passed to a feedforward network with a softmax over the two classes (IsNext and NotNext).
  - The input embeddings are passed through multiple encoder layers to output some final hidden states.


- In this notebook, for application purposes, we will be revisiting the reddit classificaton task that you saw earlier. Except this time, instead of a non-NLP classifier, we will be using our own NLP classifier. Specifically, we will be training AND fine-tuning our own NLP classifier!!

### Fine-tuning a pretrained model
- To use the pretrained model for text classification, we will add a sequence classification head on top of the base BERT model's architecture. The sequence classification head is a linear layer that accepts the final hidden states and performs a linear transformation to convert them into logits (which can then be normalized through softmax). The cross-entropy loss is the loss function used between the logits and target while training the model, to find the most likely label!

Now, lets get started! First, we need to import our dataset.

In [2]:
import pandas as pd

df = pd.read_csv("../classification/reddit_story_niche_classification_dataset/reddit_posts_with_niches_large_with_features.csv")

df.head(5)

,title,selftext,subreddit,flair,score,num_comments,upvote_ratio,created_utc,id,url,niche,title_length,contains_question,contains_capslock,engagement_score,hour_of_posting,selftext_length
0,Stan Lee has passed away at 95 years old,As many of you know today is day that many of ...,AskReddit,Breaking News,175369,27635,0.87,1.542052e+09,9whgf4,https://www.reddit.com/r/AskReddit/comments/9w...,informative,9,0,1,180206.03,19,76
1,Professor Stephen Hawking has passed away at t...,We have lost one of the greatest minds in hist...,AskReddit,Breaking News,117206,2688,0.84,1.521002e+09,84anfy,https://www.reddit.com/r/AskReddit/comments/84...,informative,11,0,1,101141.04,4,158
2,Suicide Prevention Megathread,With the news today of the passing of the amaz...,AskReddit,Modpost,104346,15803,0.82,1.528472e+09,8pks1u,https://www.reddit.com/r/AskReddit/comments/8p...,informative,3,0,1,101366.72,15,159
3,"Ruth Bader Ginsburg, US Supreme Court Justice,...","As many of you know, today [Ruth Bader Ginsbur...",AskReddit,Breaking News,99515,10307,0.82,1.600476e+09,ivici8,https://www.reddit.com/r/AskReddit/comments/iv...,informative,10,0,1,91909.30,0,70
4,I can’t breathe. Black lives matter.,As the gap of the political divide in our worl...,AskReddit,Modpost,96755,6705,0.79,1.591143e+09,gvj9a9,https://www.reddit.com/r/AskReddit/comments/gv...,informative,6,0,1,83141.45,0,396


This task is going to simply revolve JUST NLP, so we are ONLY going to look at the post's title and main body text. We will not be looking at anything else in the dataset.

So for this purpose, we will have to clean our dataset.

In [11]:
df = df.drop(columns=[c for c in list(df.columns) if c not in ["selftext", "title", "niche"]])
# We keep selftext and title because those are our features, and the niche is obviously our label

df.head(5)

,title,selftext,niche
0,Stan Lee has passed away at 95 years old,As many of you know today is day that many of ...,informative
1,Professor Stephen Hawking has passed away at t...,We have lost one of the greatest minds in hist...,informative
2,Suicide Prevention Megathread,With the news today of the passing of the amaz...,informative
3,"Ruth Bader Ginsburg, US Supreme Court Justice,...","As many of you know, today [Ruth Bader Ginsbur...",informative
4,I can’t breathe. Black lives matter.,As the gap of the political divide in our worl...,informative
